# 족보 문제 추출 — 아이폰 / 아이패드용

컴퓨터 없이 사파리에서만 돌아갑니다. 족보는 **본인 드라이브에 그대로 둔 채** 인증으로 읽으므로 공개로 바꿀 필요가 없습니다.

**쓰는 법**: 셀 왼쪽의 ▶ 버튼을 위에서부터 차례로 누르면 됩니다.

| 셀 | 하는 일 | 매번? |
|---|---|---|
| 1 | 설치 | 세션마다 (약 1분) |
| 2 | 드라이브 연결 | 세션마다 |
| 3 | 족보 폴더 찾기 | 처음 1회 |
| 4 | 문제 은행 만들기 | 족보 바뀔 때만 |
| 5 | 후보 뽑기 | 강의안마다 |
| 6 | Claude로 판정 | 강의안마다 |
| 7 | PDF 만들기 | 강의안마다 |

## 1. 설치

In [ ]:
!apt-get -qq install -y fonts-nanum > /dev/null 2>&1   # 한글 폰트 (결과 PDF 품질)
!rm -rf /content/Jokbo-extractor
!git clone -q -b claude/program-analysis-lecture-extraction-9m9bm7 \
    https://github.com/ysh030812-collab/Jokbo-extractor.git /content/Jokbo-extractor
!pip install -q pypdf reportlab python-docx pymupdf
%cd /content/Jokbo-extractor
print('\n설치 완료')

## 2. 구글 드라이브 연결

실행하면 계정 선택 창이 뜹니다. 허용해 주세요. **파일이 어디로 복사되지 않고, 코랩이 읽기만 합니다.**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. 족보 폴더 찾기 (처음 1회)

아래를 실행하면 `족보` 폴더 후보를 찾아 줍니다. 나온 경로를 4번 셀의 `JOKBO_DIR` 에 붙여넣으세요.

In [ ]:
import os

roots = [p for p in ('/content/drive/MyDrive', '/content/drive/Shareddrives',
                     '/content/drive/Shared drives') if os.path.isdir(p)]
hits = []
for root in roots:
    for dirpath, dirnames, filenames in os.walk(root):
        if dirpath.count(os.sep) - root.count(os.sep) > 6:
            dirnames[:] = []                     # 너무 깊이 들어가지 않는다
            continue
        pdfs = [f for f in filenames if '풀이' in f and f.lower().endswith('.pdf')]
        if pdfs:
            hits.append((dirpath, len(pdfs)))

if hits:
    print('족보 폴더 후보:\n')
    for d, n in sorted(hits, key=lambda x: -x[1]):
        print(f'  ({n}개 풀이 PDF)  {d}')
else:
    print('못 찾았습니다. 아래를 직접 훑어보세요:')
    for root in roots:
        print(' ', root, '->', os.listdir(root)[:20])

## 4. 문제 은행 만들기

`JOKBO_DIR` 을 3번에서 나온 경로로 바꾸고 실행하세요. **연도별 문항 수가 실제 시험과 비슷한지 꼭 확인하세요.**

In [ ]:
JOKBO_DIR = '/content/drive/MyDrive/본1-1/4. 감염과 면역 Ⅰ/2. 기말/족보'   # <- 여기 수정

import os
assert os.path.isdir(JOKBO_DIR), f'경로가 없습니다: {JOKBO_DIR}'
os.environ['JOKBO_DIR'] = JOKBO_DIR
!python jokbo.py index --library "$JOKBO_DIR"

## 5. 강의안으로 후보 뽑기

`LECTURE` 에 강의안 PDF 경로를 넣으세요. 보통 `족보` 폴더의 **한 단계 위**에 있습니다.

In [ ]:
LECTURE = os.path.join(os.path.dirname(JOKBO_DIR), '73. Antiviral agents (김채균).pdf')   # <- 여기 수정
SUBJECT = '감면'

assert os.path.exists(LECTURE), (
    f'강의안이 없습니다: {LECTURE}\n\n같은 폴더 목록:\n  '
    + '\n  '.join(sorted(os.listdir(os.path.dirname(LECTURE)))[:40]))
os.environ['LECTURE'] = LECTURE
os.environ['SUBJECT'] = SUBJECT
!python jokbo.py search "$LECTURE" --subject "$SUBJECT" --top 40 --max-chars 700 > /content/후보.json
print(open('/content/후보.json', encoding='utf-8').read())

## 6. Claude 앱에서 판정하기

5번 출력을 **길게 눌러 전체 복사** → 아이폰/아이패드의 **Claude 앱**을 열고,
**강의안 PDF를 첨부**한 뒤 아래 문장과 함께 붙여넣으세요.

> 첨부한 강의안으로 **풀 수 있는** 문제만 골라줘. 단어가 겹치는지가 아니라, 강의안 내용만으로 답을 고를 근거가 있는지로 판단해줘.
> 각 문제마다 solvable / partial / unrelated 와 근거가 된 강의안 쪽수를 적어줘.
> 마지막에 solvable 만 모아서 이 형식의 JSON 한 줄로 만들어줘:
> `[{"year":2023,"term":"기말","q":[6,7,8]}]`
>
> (여기에 후보.json 내용 붙여넣기)

Claude가 마지막에 준 JSON 한 줄을 복사해서 7번 셀에 붙여넣으면 됩니다.

## 7. PDF 만들기

In [ ]:
SELECT = '[{"year":2023,"term":"기말","q":[6,7,8]}]'        # <- Claude가 준 JSON 붙여넣기
OUT    = '/content/drive/MyDrive/족보추출/73_antiviral.pdf'   # <- 저장 위치 (드라이브)

os.makedirs(os.path.dirname(OUT), exist_ok=True)
os.environ['SELECT'] = SELECT
os.environ['OUT'] = OUT
!python jokbo.py build --subject "$SUBJECT" --library "$JOKBO_DIR" --out "$OUT" --select "$SELECT"

from google.colab import files
print('\n드라이브에 저장됨:', OUT)
print('아이패드 "파일" 또는 "Google Drive" 앱에서 바로 열 수 있습니다.')
files.download(OUT)     # 기기에 바로 내려받기도 함

---
## 막히면

| 증상 | 해결 |
|---|---|
| 3번에서 아무것도 안 나옴 | 족보가 **공유 드라이브**에 있을 수 있습니다. 출력 마지막의 폴더 목록을 보고 직접 경로를 찾으세요 |
| 4번에서 특정 연도가 0문제 | 그 파일 구조가 다릅니다. 출력 전체를 Claude에게 보여주세요 |
| 5번 `텍스트를 추출하지 못했습니다` | 스캔 이미지 강의안입니다. 5번을 건너뛰고, 6번에서 강의안 PDF만 첨부해 Claude에게 직접 물어보세요 |
| 세션이 끊김 | 코랩은 일정 시간 후 초기화됩니다. 1·2번부터 다시 실행하면 됩니다 (족보는 드라이브에 그대로 있습니다) |